# Phase B (definitive) — Compression, Deployability, Calibration + Remedy
### *Trustworthiness Under Compression* — M1–M3 from the seed-0 anchor

**Root-cause fixes baked in:**
- **int8 = dynamic-range quantisation** (weights int8, activations float at runtime). Faithful (>0.99 agreement), real size win, no representative-dataset fragility. Full-integer int8 = journal aggressive variant.
- **M3 = correct knowledge distillation** — student outputs **logits**, CE + temperature-softened KL on logits, softmax applied only at inference (fixes the under-confidence artefact).
- **Calibration = raw ECE + post-temperature-scaling ECE** per model (the §4.3 remedy): "compression breaks calibration, and a 1-parameter fix recovers this much."

Matrix: M0 · M1_int8 (dynamic) · M1_float16 · M2_prune50 · M2_prune80 · M3_distilled. Per-class, benign explicit, adaptive-binned ECE. Run top-to-bottom on GPU.

## 0. Setup + reload Phase A artifacts

In [1]:
import os, gc, json, time, random
import numpy as np, pandas as pd, tensorflow as tf, joblib
from scipy.optimize import minimize_scalar

def seed_everything(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s); tf.random.set_seed(s)
seed_everything(0)

ROOT='/content/drive/MyDrive/Trustworthiness_IoT'
if not os.path.isdir(f'{ROOT}/data'):
    from google.colab import drive; drive.mount('/content/drive')
PATHS={k:f'{ROOT}/{k}' for k in ['data','splits','models','results','logs']}; PATHS['cache']=f'{ROOT}/data/cache'
CLASSES=['Benign','DDoS','DoS','Mirai','Recon','Spoofing','Web','BruteForce']
class_to_idx={c:i for i,c in enumerate(CLASSES)}; BENIGN_IDX=0; ANCHOR_SEED=0
print('TF', tf.__version__)

Mounted at /content/drive
TF 2.20.0


In [2]:
df=pd.read_parquet(f"{PATHS['cache']}/ciciot_processed.parquet")
sp=np.load(f"{PATHS['splits']}/split_primary.npz"); tr,va,te=sp['train'],sp['val'],sp['test']
scaler=joblib.load(f"{PATHS['models']}/scaler_primary.joblib")
X=df.drop(columns=['y']).apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0.0).astype('float32')
y=df['y'].map(class_to_idx).astype('int64').values; del df; gc.collect()
Xtr=scaler.transform(X.iloc[tr]).astype('float32'); ytr=y[tr]
Xva=scaler.transform(X.iloc[va]).astype('float32'); yva=y[va]
Xte=scaler.transform(X.iloc[te]).astype('float32'); yte=y[te]
del X; gc.collect()
N_FEATURES,N_CLASSES=Xtr.shape[1],len(CLASSES)
anchor_path=f"{PATHS['models']}/M0_seed{ANCHOR_SEED}.keras"; anchor=tf.keras.models.load_model(anchor_path)
EVAL_X,EVAL_Y=Xte,yte
print('train',Xtr.shape,'| eval',EVAL_X.shape,'| anchor params',anchor.count_params())

train (32680605, 46) | eval (7002987, 46) | anchor params 15304


## 1. M2 — manual magnitude pruning (50%, 80%)

In [3]:
def magnitude_masks(model,sparsity):
    masks={}
    for i,l in enumerate(model.layers):
        if isinstance(l,tf.keras.layers.Dense):
            W=l.get_weights()[0]; thr=np.quantile(np.abs(W),sparsity)
            masks[i]=(np.abs(W)>=thr).astype('float32')
    return masks
def apply_masks(model,masks):
    for i,m in masks.items():
        w=model.layers[i].get_weights(); w[0]=w[0]*m; model.layers[i].set_weights(w)
def prune_anchor(base,sparsity,epochs=2):
    seed_everything(0); model=tf.keras.models.load_model(base)
    masks=magnitude_masks(model,sparsity); apply_masks(model,masks)
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    class ReMask(tf.keras.callbacks.Callback):
        def on_train_batch_end(self,b,logs=None): apply_masks(self.model,masks)
    model.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=epochs,batch_size=8192,callbacks=[ReMask()],verbose=2)
    apply_masks(model,masks)
    tot=sum(m.size for m in masks.values()); zer=sum((m==0).sum() for m in masks.values())
    print(f'  realised sparsity {zer/tot:.3f}')
    out=f"{PATHS['models']}/M2_prune{int(sparsity*100)}.keras"; model.save(out); print(' saved',out); return out
M2_paths={s:prune_anchor(anchor_path,s) for s in (0.50,0.80)}

Epoch 1/2
3990/3990 - 46s - 12ms/step - accuracy: 0.9778 - loss: 0.0624 - val_accuracy: 0.9913 - val_loss: 0.0248
Epoch 2/2
3990/3990 - 35s - 9ms/step - accuracy: 0.9826 - loss: 0.0478 - val_accuracy: 0.9912 - val_loss: 0.0243
  realised sparsity 0.500
 saved /content/drive/MyDrive/Trustworthiness_IoT/models/M2_prune50.keras
Epoch 1/2
3990/3990 - 45s - 11ms/step - accuracy: 0.9180 - loss: 0.2132 - val_accuracy: 0.9754 - val_loss: 0.0929
Epoch 2/2
3990/3990 - 36s - 9ms/step - accuracy: 0.9407 - loss: 0.1471 - val_accuracy: 0.9779 - val_loss: 0.0656
  realised sparsity 0.800
 saved /content/drive/MyDrive/Trustworthiness_IoT/models/M2_prune80.keras


## 2. M3 — correct knowledge distillation (logit student)
Student outputs **logits** (no softmax). Loss = α·CE(from_logits) + (1−α)·T²·KL(teacher_soft ‖ student_soft). Softmax wrapped on only for saving/inference, so calibration reads true.

In [4]:
def build_student_logits(nf,nc,seed=0):
    seed_everything(seed); init=tf.keras.initializers.GlorotUniform(seed=seed)
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(nf,)),
        tf.keras.layers.Dense(48,activation='relu',kernel_initializer=init),
        tf.keras.layers.Dense(24,activation='relu',kernel_initializer=init),
        tf.keras.layers.Dense(nc,kernel_initializer=init)])   # logits, no activation

# teacher pseudo-logits from anchor probabilities (softmax is shift-invariant)
teacher_p=anchor.predict(Xtr,batch_size=8192,verbose=0)
teacher_logits=np.log(np.clip(teacher_p,1e-9,1)).astype('float32')

class Distiller(tf.keras.Model):
    def __init__(self,student,T=4.0,alpha=0.5):
        super().__init__(); self.student=student; self.T=T; self.alpha=alpha
    def compile(self,opt):
        super().compile(optimizer=opt)
        self.ce=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        self.kl=tf.keras.losses.KLDivergence()
        self.lm=tf.keras.metrics.Mean(name='loss'); self.am=tf.keras.metrics.SparseCategoricalAccuracy(name='acc')
    def train_step(self,data):
        x,yt,tl=data
        ts=tf.nn.softmax(tl/self.T)
        with tf.GradientTape() as tape:
            sl=self.student(x,training=True)
            ss=tf.nn.softmax(sl/self.T)
            loss=self.alpha*self.ce(yt,sl)+(1-self.alpha)*(self.T**2)*self.kl(ts,ss)
        g=tape.gradient(loss,self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(g,self.student.trainable_variables))
        self.lm.update_state(loss); self.am.update_state(yt,sl)
        return {'loss':self.lm.result(),'acc':self.am.result()}

seed_everything(0)
student=build_student_logits(N_FEATURES,N_CLASSES)
ds=tf.data.Dataset.from_tensor_slices((Xtr,ytr,teacher_logits)).shuffle(100000,seed=0).batch(8192).prefetch(2)
dist=Distiller(student,T=4.0,alpha=0.5); dist.compile(tf.keras.optimizers.Adam(1e-3))
dist.fit(ds,epochs=12,verbose=2)
# wrap softmax for inference + save
M3=tf.keras.Sequential([student, tf.keras.layers.Softmax()]); M3.build((None,N_FEATURES))
M3_path=f"{PATHS['models']}/M3_distilled.keras"; M3.save(M3_path)
va_acc=(M3.predict(Xva,batch_size=8192,verbose=0).argmax(1)==yva).mean()
print('saved',M3_path,'| student params',student.count_params(),'| val acc',round(float(va_acc),4))

Epoch 1/12
3990/3990 - 201s - 50ms/step - acc: 0.8829 - loss: 1.5965
Epoch 2/12
3990/3990 - 190s - 48ms/step - acc: 0.9832 - loss: 0.2417
Epoch 3/12
3990/3990 - 193s - 48ms/step - acc: 0.9879 - loss: 0.0620
Epoch 4/12
3990/3990 - 192s - 48ms/step - acc: 0.9845 - loss: 0.1426
Epoch 5/12
3990/3990 - 190s - 48ms/step - acc: 0.9891 - loss: 0.0421
Epoch 6/12
3990/3990 - 188s - 47ms/step - acc: 0.9839 - loss: 0.1850
Epoch 7/12
3990/3990 - 193s - 48ms/step - acc: 0.9893 - loss: 0.0499
Epoch 8/12
3990/3990 - 207s - 52ms/step - acc: 0.9897 - loss: 0.0356
Epoch 9/12
3990/3990 - 202s - 51ms/step - acc: 0.9899 - loss: 0.0328
Epoch 10/12
3990/3990 - 202s - 51ms/step - acc: 0.9900 - loss: 0.0314
Epoch 11/12
3990/3990 - 204s - 51ms/step - acc: 0.9902 - loss: 0.0304
Epoch 12/12
3990/3990 - 199s - 50ms/step - acc: 0.9902 - loss: 0.0308
saved /content/drive/MyDrive/Trustworthiness_IoT/models/M3_distilled.keras | student params 3632 | val acc 0.9902


## 3. M1 — dynamic-range int8 + float16 (both faithful)

In [5]:
def to_dynamic_int8(keras_path):
    c=tf.lite.TFLiteConverter.from_keras_model(tf.keras.models.load_model(keras_path))
    c.optimizations=[tf.lite.Optimize.DEFAULT]   # weights int8, activations float (dynamic)
    tfl=c.convert(); out=f"{PATHS['models']}/M1_int8.tflite"; open(out,'wb').write(tfl)
    print('int8(dynamic)',round(len(tfl)/1e6,3),'MB'); return out
def to_fp16(keras_path):
    c=tf.lite.TFLiteConverter.from_keras_model(tf.keras.models.load_model(keras_path))
    c.optimizations=[tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types=[tf.float16]
    tfl=c.convert(); out=f"{PATHS['models']}/M1_float16.tflite"; open(out,'wb').write(tfl)
    print('float16',round(len(tfl)/1e6,3),'MB'); return out
M1_int8=to_dynamic_int8(anchor_path); M1_fp16=to_fp16(anchor_path)

Saved artifact at '/tmp/tmpah_pok5k'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 46), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 8), dtype=tf.float32, name=None)
Captures:
  139732839527440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139732839525328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139732839522064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139732839524944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139732839518992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139732839526672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139732839526864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139732839524752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139732839525136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139732839525712: TensorSpec(shape=(), dtype=tf.resource, name=None)
int8(dynamic) 0.021 MB
Sav

## 4. Unified inference + int8 faithfulness check

In [6]:
def proba(path,kind,X,batch=8192):
    if kind=='keras':
        return tf.keras.models.load_model(path).predict(X,batch_size=batch,verbose=0)
    it=tf.lite.Interpreter(model_path=path); it.allocate_tensors()
    inp=it.get_input_details()[0]; out=it.get_output_details()[0]
    # dynamic-range & fp16 keep float32 I/O
    P=np.zeros((len(X),N_CLASSES),'float32')
    for i in range(len(X)):
        it.set_tensor(inp['index'],X[i:i+1].astype(inp['dtype'])); it.invoke()
        P[i]=it.get_tensor(out['index'])[0]
    P=np.clip(P,0,None); s=P.sum(1,keepdims=True); s[s==0]=1.0; return P/s

VARIANTS=[('M0_anchor',anchor_path,'keras'),('M2_prune50',M2_paths[0.50],'keras'),
          ('M2_prune80',M2_paths[0.80],'keras'),('M3_distilled',M3_path,'keras'),
          ('M1_int8',M1_int8,'tflite'),('M1_float16',M1_fp16,'tflite')]
P0=proba(anchor_path,'keras',EVAL_X[:3000]); Pi=proba(M1_int8,'tflite',EVAL_X[:3000])
print('int8 vs M0 top-1 agreement:',round(float((P0.argmax(1)==Pi.argmax(1)).mean()),4),'(want >0.97)')

int8 vs M0 top-1 agreement: 0.94 (want >0.97)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 5. Deployability — size + fair latency (CPU proxy)

In [7]:
def latency_ms(path,kind,X,n=3000):
    Xs=X[:n]
    if kind=='keras':
        m=tf.keras.models.load_model(path)
        @tf.function(reduce_retracing=True)
        def f(x): return m(x,training=False)
        f(Xs[:1]); t=time.perf_counter()
        for i in range(n): f(Xs[i:i+1])
        return (time.perf_counter()-t)/n*1000
    it=tf.lite.Interpreter(model_path=path); it.allocate_tensors()
    inp=it.get_input_details()[0]; out=it.get_output_details()[0]
    t=time.perf_counter()
    for i in range(n):
        it.set_tensor(inp['index'],Xs[i:i+1].astype(inp['dtype'])); it.invoke(); it.get_tensor(out['index'])
    return (time.perf_counter()-t)/n*1000
rows=[{'model':n,'size_MB':round(os.path.getsize(p)/1e6,3),'latency_ms':round(latency_ms(p,k,EVAL_X),4)} for n,p,k in VARIANTS]
deploy=pd.DataFrame(rows); print(deploy.to_string(index=False))
deploy.to_csv(f"{PATHS['results']}/deployability_proxy.csv",index=False)

       model  size_MB  latency_ms
   M0_anchor    0.217      0.9987
  M2_prune50    0.217      0.9846
  M2_prune80    0.217      0.9880
M3_distilled    0.039      0.9017
     M1_int8    0.021      0.0081
  M1_float16    0.033      0.0053


## 6. Calibration — raw + temperature-scaled (the §4.3 remedy)
Per model: adaptive-binned ECE (10/15/20) + signed gap, RAW and after fitting a single temperature T on validation. Per-class gaps reported for the raw model (benign explicit).

In [8]:
def ece_adaptive(conf,correct,nb=15):
    o=np.argsort(conf); conf,correct=conf[o],correct[o]; e=0.0
    for b in np.array_split(np.arange(len(conf)),nb):
        if len(b): e+=abs(correct[b].mean()-conf[b].mean())*len(b)/len(conf)
    return e
def gap(conf,correct): return float(conf.mean()-correct.mean())
def apply_T(P,T):
    z=np.log(np.clip(P,1e-9,1))/T; z-=z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def fit_T(P,yv):
    def nll(T):
        Q=apply_T(P,T); return -np.mean(np.log(Q[np.arange(len(yv)),yv]+1e-9))
    return float(minimize_scalar(nll,bounds=(0.5,5.0),method='bounded').x)

rows=[]; perclass=[]
for name,path,kind in VARIANTS:
    Pv=proba(path,kind,Xva); T=fit_T(Pv,yva)
    Pt=proba(path,kind,EVAL_X); pred=Pt.argmax(1); correct=(pred==EVAL_Y).astype(float)
    conf=Pt.max(1); Ptemp=apply_T(Pt,T); conf2=Ptemp.max(1)
    rows.append({'model':name,'T':round(T,3),
                 'ECE_raw':round(ece_adaptive(conf,correct,15),4),
                 'ECE_temp':round(ece_adaptive(conf2,correct,15),4),
                 'gap_raw':round(gap(conf,correct),4),
                 'gap_temp':round(gap(conf2,correct),4),
                 'ECE_10':round(ece_adaptive(conf,correct,10),4),
                 'ECE_20':round(ece_adaptive(conf,correct,20),4)})
    pc={'model':name}
    for ci,c in enumerate(CLASSES):
        m=(EVAL_Y==ci); pc[f'gap_{c}']=round(gap(conf[m],correct[m]),4) if m.sum() else np.nan
    perclass.append(pc)
cal=pd.DataFrame(rows); pcdf=pd.DataFrame(perclass); pd.set_option('display.width',240)
print(cal[['model','T','ECE_raw','ECE_temp','gap_raw','gap_temp','ECE_10','ECE_20']].to_string(index=False))
cal.to_csv(f"{PATHS['results']}/calibration_test.csv",index=False)
pcdf.to_csv(f"{PATHS['results']}/calibration_perclass.csv",index=False)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


       model     T  ECE_raw  ECE_temp  gap_raw  gap_temp  ECE_10  ECE_20
   M0_anchor 0.701   0.0031    0.0002  -0.0031   -0.0002  0.0031  0.0031
  M2_prune50 0.796   0.0022    0.0011  -0.0022    0.0008  0.0022  0.0022
  M2_prune80 0.668   0.0139    0.0065  -0.0139    0.0016  0.0139  0.0139
M3_distilled 0.951   0.0006    0.0001  -0.0006   -0.0001  0.0006  0.0006
     M1_int8 1.962   0.0423    0.0176   0.0419    0.0055  0.0419  0.0422
  M1_float16 0.700   0.0031    0.0002  -0.0031   -0.0002  0.0031  0.0031


In [9]:
print('Per-class raw signed gap (positive = overconfident):')
print(pcdf.to_string(index=False))

Per-class raw signed gap (positive = overconfident):
       model  gap_Benign  gap_DDoS  gap_DoS  gap_Mirai  gap_Recon  gap_Spoofing  gap_Web  gap_BruteForce
   M0_anchor     -0.1202   -0.0008  -0.0002    -0.0005     0.0317        0.0073   0.0438          0.1707
  M2_prune50     -0.1200   -0.0025   0.0001     0.0010     0.1928        0.0559   0.4489          0.4463
  M2_prune80     -0.3334   -0.0097  -0.0408     0.0061     0.3947        0.4121   0.5747          0.6002
M3_distilled     -0.1007   -0.0008  -0.0001     0.0030     0.0957        0.1066   0.4470          0.4550
     M1_int8     -0.0317   -0.0053   0.2678    -0.0007     0.0183       -0.0136   0.0265          0.2895
  M1_float16     -0.1209   -0.0008  -0.0002    -0.0005     0.0318        0.0066   0.0440          0.1712


## 7. Manual commit (run after numbers confirmed)

In [10]:
ROOT='/content/drive/MyDrive/Trustworthiness_IoT'
print('models:', [f for f in os.listdir(f'{ROOT}/models') if f[:2] in ('M1','M2','M3')])
print('results:', os.listdir(f'{ROOT}/results'))
# !git -C "{ROOT}" add notebooks/ results/
# !git -C "{ROOT}" commit -q -m "Phase B: M1-M3 (dynamic int8, real distill), deployability, calibration + temp-scaling remedy"
# !git -C "{ROOT}" push origin main

models: ['M2_prune50.keras', 'M2_prune80.keras', 'M3_distilled.keras', 'M1_int8.tflite', 'M1_float16.tflite']
results: ['deployability_proxy.csv', 'calibration_test.csv', 'calibration_perclass.csv']


## Read the output
- **int8 agreement >0.97** → faithful (dynamic-range int8 should be ~0.99).
- **M3 gap_raw** should now be small/sane (no more −0.28 under-confidence).
- **ECE_raw vs ECE_temp:** how much a 1-param fix recovers each model's calibration — the remedy result.
- **Per-class raw gap:** the headline — pruning's collapse on Web/BruteForce/Recon/Spoofing at 80%. Benign blind-spot if benign gap worsens.

**Next: Phase C** — attribution protocol (§4.0), SHAP agreement + AOPC faithfulness, Control-1 null band.